In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

# Loading laptop data
file_path = "../data/laptopData.csv"
df = pd.read_csv(file_path)

# Loading spec data
spec_file_path = "../data/specData.csv"
spec_df = pd.read_csv(spec_file_path)

df.head()

In [ ]:
# Making sure of dataframe "structure"
print("Laptop Data Shape:", df.shape)
print("\nLaptop Columns:\n", df.columns)
print("\nLaptop Data types:\n", df.dtypes)
print("\nLaptop Missing values:\n", df.isna().sum())

print("\n" + "="*60)

print("Spec Data Shape:", spec_df.shape)
print("\nSpec Columns:\n", spec_df.columns)
print("\nSpec Data types:\n", spec_df.dtypes)
print("\nSpec Missing values:\n", spec_df.isna().sum())

In [ ]:
# Dropping unnecessary columns: the "index" column
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

if "Unnamed: 0" in spec_df.columns:
    spec_df = spec_df.drop(columns=["Unnamed: 0"])

In [ ]:
# Removing duplicate values
df = df.drop_duplicates()
spec_df = spec_df.drop_duplicates()

print("After removing duplicates, laptop data:", df.shape)
print("After removing duplicates, spec data:", spec_df.shape)

In [ ]:
## Cleaning numeric columns: the "RAM" and "Weight" column have numeric values/information,
## but they're stored as strngs because the units are included (gigabytes of RAM, or 
## kilogrms of weight); so I'm removing the units.

# RAM: remove "GB"
df["Ram"] = df["Ram"].astype(str).str.replace("GB", "", regex=False).str.strip()
df["Ram"] = pd.to_numeric(df["Ram"], errors="coerce")

# Weight: remove "kg"
df["Weight"] = df["Weight"].astype(str).str.replace("kg", "", regex=False).str.strip()
df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
    # "pd.to_numeric(errors="coerce")" -- converts values to numbers and any vaules that 
    # cannot be converted are turned into NaN (for less errors); makes data cleaning safer
    # because it prevents crashes, and I'll handle these later

# CHeck if any mssing values of RAM or Weight
print(df[["Ram", "Weight"]].isna().sum())

# Handleing missing values by 'replacing' them with the median value
## Median is less affected by outliers; I am assuming the laptop specs are not evenly 
## distributed (most laptops are 8 or 16 GB RAM, but some could be 32 or 64 GB, making the 
## data right-skewed). It also maked no sense to use the mean as it might result in a 
## number that doesn't actually wokr for the data (mean could be something that is not euqal
## to (2^n), which is not possible for RAM)
df["Ram"] = df["Ram"].fillna(df["Ram"].median())
df["Weight"] = df["Weight"].fillna(df["Weight"].median())

# Make RAM integer
df["Ram"] = df["Ram"].astype(int)

# CHeck if any mssing values of RAM or Weight after filling them with the mean
print(df[["Ram", "Weight"]].isna().sum())

In [ ]:
## The data set had CPU speeds as strings (like above); "Intel Core i5 2.3GHz" wouldn't have
## been useful for the model, as ML models learn relationships between features (like numeric
## inputs) and output (price), so we need the numbers separate from the core/type

# Getting the CPU speed (GHz)
df["CPU_speed"] = df["Cpu"].astype(str).str.extract(r'(\d+\.?\d*)GHz')
df["CPU_speed"] = pd.to_numeric(df["CPU_speed"], errors="coerce")


# Getting the CPU 'core'/type (Intel, AMD (Ryzen and others), and others)
def extract_cpu_core(cpu):
    cpu = str(cpu).lower()

    # Intel
    if "i3" in cpu:
        return "i3"
    elif "i5" in cpu:
        return "i5"
    elif "i7" in cpu:
        return "i7"
    elif "i9" in cpu:
        return "i9"

    # AMD Ryzen
    elif "ryzen 3" in cpu or "ryzen3" in cpu:
        return "ryzen3"
    elif "ryzen 5" in cpu or "ryzen5" in cpu:
        return "ryzen5"
    elif "ryzen 7" in cpu or "ryzen7" in cpu:
        return "ryzen7"
    elif "ryzen 9" in cpu or "ryzen9" in cpu:
        return "ryzen9"

    # Other AMD families seen in the laptop dataset
    elif "a4" in cpu:
        return "amd_a4"
    elif "a6" in cpu:
        return "amd_a6"
    elif "a8" in cpu:
        return "amd_a8"
    elif "a9" in cpu:
        return "amd_a9"
    elif "a10" in cpu:
        return "amd_a10"
    elif "a12" in cpu:
        return "amd_a12"
    elif "e-series" in cpu or re.search(r'\be2\b', cpu):
        return "amd_e"
    elif "fx" in cpu:
        return "amd_fx"
    elif "athlon" in cpu:
        return "athlon"

    else:
        return "other"

import re

df["CPU_core"] = df["Cpu"].apply(extract_cpu_core)

# Chekc if any missing CPU speed values
print("Missing CPU_speed before filling:", df["CPU_speed"].isna().sum())

# Fill missing values
df["CPU_core"] = df["CPU_core"].fillna("other")
df["CPU_speed"] = df.groupby("CPU_core")["CPU_speed"].transform(lambda x: x.fillna(x.median()))
    # using median again because it'll give more realistic values/ the typical values
df["CPU_speed"] = df["CPU_speed"].fillna(df["CPU_speed"].median())
    # for groups where all values were missing, fill any missing CPU values with the overall median

# Chekc if any missing CPU speed values, and count of cpu core types
print("Missing CPU_speed after fill:", df["CPU_speed"].isna().sum())
print(df["CPU_core"].value_counts())

In [ ]:
## The data set had the screen resoluton stored as strings, which wouldn't be useful for the
## model (same reason as above), so getting the resolution only

# Getting the resolution
resolution = df["ScreenResolution"].astype(str).str.extract(r'(\d+)x(\d+)')

df["Resolution_X"] = pd.to_numeric(resolution[0], errors="coerce")
df["Resolution_Y"] = pd.to_numeric(resolution[1], errors="coerce")

# Fill missing values using the median
df["Resolution_X"] = df["Resolution_X"].fillna(df["Resolution_X"].median())
df["Resolution_Y"] = df["Resolution_Y"].fillna(df["Resolution_Y"].median())

# Getting the total number of pixels
df["Pixel_Count"] = df["Resolution_X"] * df["Resolution_Y"]

# Check for any missing resolution values
print("Missing Resolution_X:", df["Resolution_X"].isna().sum())
print("Missing Resolution_Y:", df["Resolution_Y"].isna().sum())
print(df[["Resolution_X", "Resolution_Y", "Pixel_Count"]].head())

In [ ]:
## SSD and HDD from string to numeric (like above/for the same reasons)

# Extract SSD and HDD sizes for GB and TB
ssd_extracted = df["Memory"].astype(str).str.extract(r'(\d+)(GB|TB)\s*SSD')
hdd_extracted = df["Memory"].astype(str).str.extract(r'(\d+)(GB|TB)\s*HDD')

# Convert size and units
def convert_storage(row):
    value, unit = row[0], row[1]
    if pd.isna(value):
        return np.nan
    value = float(value)
    if unit == "TB":
        return value * 1024
    return value

df["SSD"] = ssd_extracted.apply(convert_storage, axis=1)
df["HDD"] = hdd_extracted.apply(convert_storage, axis=1)

# Replace NaN with 0, meaning no SSD/HDD was in the data
df["SSD"] = df["SSD"].fillna(0)
df["HDD"] = df["HDD"].fillna(0)

# Check for any missing SSD and HDD values
print("Missing SSD:", df["SSD"].isna().sum())
print("Missing HDD:", df["HDD"].isna().sum())

print(df[["SSD", "HDD"]].head())

In [ ]:
## Separating GPU brand/type (like above/for the same reasons)

def extract_gpu_brand(gpu):
    gpu = str(gpu).lower()
    if "nvidia" in gpu:
        return "nvidia"
    elif "amd" in gpu or "radeon" in gpu:
        return "amd"
    elif "intel" in gpu:
        return "intel"
    else:
        return "other"

df["GPU_brand"] = df["Gpu"].apply(extract_gpu_brand)
# 

# Check distribution
print(df["GPU_brand"].value_counts())

In [ ]:
## Add average spec_rating by brand from specData.csv

# Make the brand/company names ready for matching
df["Company_match"] = df["Company"].astype(str).str.strip().str.lower()
spec_df["brand_match"] = spec_df["brand"].astype(str).str.strip().str.lower()

# Make sure spec_rating is numeric
spec_df["spec_rating"] = pd.to_numeric(spec_df["spec_rating"], errors="coerce")

# Compute the average spec rating for each brand
brand_avg_spec = (
    spec_df.groupby("brand_match", as_index=False)["spec_rating"]
    .mean()
    .rename(columns={"spec_rating": "avg_brand_spec_rating"})
)

print("Average spec rating by brand:")
print(brand_avg_spec.head())

# Add the spec ratings to the cleaned (in progress lol) laptop dataframe
df = df.merge(
    brand_avg_spec,
    how="left",
    left_on="Company_match",
    right_on="brand_match"
)

# Drop the rows where the laptop company does not exist in specData.csv
before_drop = df.shape[0]
df = df.dropna(subset=["avg_brand_spec_rating"])
after_drop = df.shape[0]

print(f"Rows before dropping unmatched brands: {before_drop}")
print(f"Rows after dropping unmatched brands: {after_drop}")
print(f"Rows removed because company was not in specData.csv: {before_drop - after_drop}")

# Drop the temporary merge column from spec data
df = df.drop(columns=["brand_match"])

In [ ]:
# Check for any last/remaining missing values
print(df.isna().sum())

# Drop rows where target (Price) is missing
df = df.dropna(subset=["Price"])

# Fill numeric columns with median, excluding price
numeric_cols = df.select_dtypes(include=np.number).columns.drop("Price")

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill in any missing categorical values with "unknown"
df["Company"] = df["Company"].fillna("unknown")
df["Company"] = df["Company"].replace("", "unknown")
df["Gpu"] = df["Gpu"].fillna("unknown")
df["OpSys"] = df["OpSys"].fillna("unknown")

# Drop remaining missing rows, like categorical columns; all numeric columns are already
# filled, so now only categorical columns still have missing values. 
df = df.dropna()

# Final check
print("\nRemaining missing values:\n", df.isna().sum())
print("\nShape:", df.shape)

In [ ]:
# Final check for numerical :))
df[df.isna().any(axis=1)]

In [ ]:
# Ensure consistent categories will be generated by standardizing categorical vriables;
# this will reduce noise and prevent duplicate categories, improving model performance.
categorical_cols = ["Company", "TypeName", "Gpu", "OpSys"]

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Make the matching column ready for next steps
df["Company_match"] = df["Company_match"].astype(str).str.strip().str.lower()

In [ ]:
# Load cleaned data to a csv file
df.to_csv("../data/cleaned_laptop_data.csv", index=False)

# Another check :')
print("Cleaned data saved successfully, slay!")
df.head()